In [1]:
import os
import torch
import time
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A

class AugmentDataset(Dataset):
    def __init__(self, csv_path, image_dir, mask_dir, split='train', augment=True):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.augment = augment

    def randomHorizontalFlip(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = cv2.flip(image, 1)
            mask = cv2.flip(mask, 1)
        return image, mask

    def randomVerticalFlip(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = cv2.flip(image, 0)
            mask = cv2.flip(mask, 0)
        return image, mask

    def randomRotate90(self, image, mask, u=0.5):
        if np.random.rand() < u:
            image = np.rot90(image).copy()
            mask = np.rot90(mask).copy()
        return image, mask

    def randomHueSaturationValue(self, image, hue_shift_limit=(-30, 30),
                                  sat_shift_limit=(-5, 5),
                                  val_shift_limit=(-15, 15), u=0.5):
        if np.random.rand() < u:
            image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
            h, s, v = cv2.split(image)
            h = h.astype(np.float32)
            s = s.astype(np.float32)
            v = v.astype(np.float32)


            h += np.random.randint(hue_shift_limit[0], hue_shift_limit[1] + 1)
            s += np.random.uniform(sat_shift_limit[0], sat_shift_limit[1])
            v += np.random.uniform(val_shift_limit[0], val_shift_limit[1])

            h = np.clip(h, 0, 179).astype(np.uint8)
            s = np.clip(s, 0, 255).astype(np.uint8)
            v = np.clip(v, 0, 255).astype(np.uint8)

            image = cv2.merge((h, s, v))
            image = cv2.cvtColor(image, cv2.COLOR_HSV2RGB)
        return image

    def randomShiftScaleRotate(self, image, mask,
                               shift_limit=(-0.1, 0.1),
                               scale_limit=(-0.1, 0.1),
                               rotate_limit=(0, 0),
                               aspect_limit=(-0.1, 0.1),
                               borderMode=cv2.BORDER_CONSTANT, u=0.5):
        if np.random.rand() < u:
            height, width = image.shape[:2]
            angle = np.random.uniform(*rotate_limit)
            scale = np.random.uniform(1 + scale_limit[0], 1 + scale_limit[1])
            aspect = np.random.uniform(1 + aspect_limit[0], 1 + aspect_limit[1])
            sx = scale * aspect / (aspect ** 0.5)
            sy = scale / (aspect ** 0.5)
            dx = int(np.random.uniform(*shift_limit) * width)
            dy = int(np.random.uniform(*shift_limit) * height)

            cc = np.cos(np.radians(angle)) * sx
            ss = np.sin(np.radians(angle)) * sy
            rotation = np.array([[cc, -ss], [ss, cc]])

            box = np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)
            box -= np.array([width / 2, height / 2], dtype=np.float32)
            box = np.dot(box, rotation.T) + np.array([width / 2 + dx, height / 2 + dy], dtype=np.float32)

            mat = cv2.getPerspectiveTransform(box.astype(np.float32), np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32))
            image = cv2.warpPerspective(image, mat, (width, height), flags=cv2.INTER_LINEAR, borderMode=borderMode, borderValue=(0, 0, 0))
            mask = cv2.warpPerspective(mask, mat, (width, height), flags=cv2.INTER_NEAREST, borderMode=borderMode, borderValue=(0,))

        return image, mask
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        mask_path = os.path.join(self.mask_dir, row['maskname'])

        # Read image & mask
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)

        # Augment
        if self.augment:
            image = self.randomHueSaturationValue(image)
            image, mask = self.randomShiftScaleRotate(image, mask)
            image, mask = self.randomHorizontalFlip(image, mask)
            image, mask = self.randomVerticalFlip(image, mask)
            image, mask = self.randomRotate90(image, mask)

        # Normalize image
        image = image.astype(np.float32) / 255.0
        image = image * 3.2 - 1.6
        image = np.transpose(image, (2, 0, 1))  # HWC → CHW

        # Normalize mask
        mask = mask.astype(np.float32) / 255.0
        mask = np.expand_dims(mask, axis=0)  # (1, H, W)
        mask = (mask > 0.5).astype(np.float32)

        return torch.tensor(image), torch.tensor(mask)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet34, ResNet34_Weights

class ASPPConv(nn.Sequential):
    """Standard Conv block with dilation for ASPP"""
    def __init__(self, in_channels, out_channels, dilation):
        super(ASPPConv, self).__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

class ASPPPooling(nn.Sequential):
    """Global Average Pooling branch for ASPP"""
    def __init__(self, in_channels, out_channels):
        super(ASPPPooling, self).__init__(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        size = x.shape[-2:]
        out = super(ASPPPooling, self).forward(x)
        return F.interpolate(out, size=size, mode='bilinear', align_corners=False)

class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling Module adapted for ResNet-34 (512 input channels)"""
    def __init__(self, in_channels=512, out_channels=256, rates=[6, 12, 18]):
        super(ASPP, self).__init__()
        modules = []
        # 1. 1x1 Convolution branch
        modules.append(nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ))
        
        # 2. Three Dilated 3x3 Convolution branches
        for rate in rates:
            modules.append(ASPPConv(in_channels, out_channels, rate))
            
        # 3. Image Pooling branch
        modules.append(ASPPPooling(in_channels, out_channels))
        
        self.convs = nn.ModuleList(modules)
        
        # 4. Final Projection layer fusing all 5 branches
        self.project = nn.Sequential(
            nn.Conv2d(len(modules) * out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        res = []
        for conv in self.convs:
            res.append(conv(x))
        res = torch.cat(res, dim=1)
        return self.project(res)


class DeepLabV3PlusResNet34(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(DeepLabV3PlusResNet34, self).__init__()
        
        # 1. ENCODER: Extract backbone from ResNet-34
        weights = ResNet34_Weights.DEFAULT if pretrained else None
        base_resnet = resnet34(weights=weights)
        
        self.initial_blocks = nn.Sequential(
            base_resnet.conv1,
            base_resnet.bn1,
            base_resnet.relu
        )
        self.maxpool = base_resnet.maxpool
        
        self.layer1 = base_resnet.layer1  # Low-level features: 64 channels, 1/4 resolution
        self.layer2 = base_resnet.layer2  # 128 channels, 1/8 resolution
        self.layer3 = base_resnet.layer3  # 256 channels, 1/16 resolution
        self.layer4 = base_resnet.layer4  # High-level features: 512 channels, 1/32 resolution

        # 2. BOTTLENECK: Multi-scale context gathering via ASPP
        self.aspp = ASPP(in_channels=512, out_channels=256)

        # 3. DECODER: Feature refinement and fusion
        # Low-level projection (reduces channels from early encoder to avoid overpowering high-level features)
        self.low_level_project = nn.Sequential(
            nn.Conv2d(64, 48, kernel_size=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True)
        )
        
        # Main Decoder Refining Layers (processes fused 256 + 48 = 304 channels)
        self.decoder_head = nn.Sequential(
            nn.Conv2d(304, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

    def forward(self, x):
        input_size = x.shape[-2:]  # Keep original H, W for final upsampling
        
        # --- Encoder Forward ---
        x = self.initial_blocks(x)
        low_level_features = self.layer1(x)  # Retain for decoder connection (1/4 size)
        
        x = self.maxpool(low_level_features)
        x = self.layer2(x)
        x = self.layer3(x)
        high_level_features = self.layer4(x) # 512 channels (1/32 size)

        # --- ASPP Bottleneck ---
        aspp_features = self.aspp(high_level_features) # 256 channels
        
        # --- Decoder Forward ---
        # 1. Upsample high-level ASPP features 8x to meet low-level size (1/4 size)
        aspp_features_upsampled = F.interpolate(
            aspp_features, size=low_level_features.shape[-2:], 
            mode='bilinear', align_corners=False
        )
        
        # 2. Process low-level features
        low_features_projected = self.low_level_project(low_level_features)
        
        # 3. Concatenate along channel dimension (U-Net style channel stacking)
        fused_features = torch.cat([aspp_features_upsampled, low_features_projected], dim=1)
        
        # 4. Refine via convolution blocks
        decoder_output = self.decoder_head(fused_features)
        
        # --- Final Upsampling to Match Input Resolution ---
        output = F.interpolate(decoder_output, size=input_size, mode='bilinear', align_corners=False)
        return output

In [4]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def get_optimizer(model, lr=5e-4, weight_decay=1e-4):
    return AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def get_scheduler(optimizer, T_0=10, T_mult=2, eta_min=1e-6):
    return CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=T_0, 
    T_mult=T_mult, 
    eta_min=eta_min
)

In [5]:
def bce_dice_loss(y_pred_logits, y_true, smooth=1e-5):
    """
    y_pred_logits: Raw model outputs (NO Sigmoid applied in the architecture)
    y_true: Ground truth binary masks
    """
    # 1. Numerically stable BCE using Logits
    bce = F.binary_cross_entropy_with_logits(y_pred_logits, y_true)
    
    # 2. Apply Sigmoid safely inside the loss for Dice calculation
    y_pred_probs = torch.sigmoid(y_pred_logits)
    
    # 3. Flatten the ENTIRE batch into a single 1D vector for maximum GPU speed
    y_pred_flat = y_pred_probs.view(-1)
    y_true_flat = y_true.view(-1)
    
    # 4. Global Dice calculation
    intersection = (y_pred_flat * y_true_flat).sum()
    dice = (2. * intersection + smooth) / (y_pred_flat.sum() + y_true_flat.sum() + smooth)
    
    dice_loss = 1.0 - dice
    
    # Combined loss
    return bce + dice_loss

In [6]:
import torch
import os

def save_model(model, path, epoch, score):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    state = {
        'model_state_dict': model.state_dict(),
        'epoch': epoch,
        'score': score
    }
    torch.save(state, path)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
# === Cấu hình ===
batch_size = 16
num_epochs = 100
lr = 1e-3
log_dir = '/kaggle/working/logs'
model_path = '/kaggle/working/best_model.pth'
csv_path = '/kaggle/input/datasets/vuongtran11233/new-division/data_division.csv'
image_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/images'
mask_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/masks'

In [9]:
# === Dataloader ===
train_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='train', augment=True)
val_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='val', augment=False)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4,pin_memory=True,drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4,pin_memory=True,drop_last=True)

In [10]:
# Previous-run best model
input_model_path = '/kaggle/input/models/vngtrnml17/deeplabv3-1/pytorch/default/1/best_model_deeplabv3.pth'

In [11]:
from collections import OrderedDict

model = DeepLabV3PlusResNet34(num_classes=1)
checkpoint = torch.load(input_model_path, map_location=device)

state_dict = checkpoint['model_state_dict']
clean_state_dict = OrderedDict()

for key, value in state_dict.items():
    if key.startswith('module.'):
        clean_name = key[7:]  # 'module.' is 7 characters long
        clean_state_dict[clean_name] = value
    else:
        clean_state_dict[key] = value
        
model.load_state_dict(clean_state_dict)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 188MB/s]


<All keys matched successfully>

In [12]:
# === Model, Loss, Optimizer ===
#model = DeepLabV3PlusResNet34(num_classes=1).to(device)
if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs for parallel training!")
    model = nn.DataParallel(model)
model.to(device)
optimizer = get_optimizer(model, lr)
scheduler = get_scheduler(optimizer)
EarlyStopPatience = 15

🚀 Using 2 GPUs for parallel training!


In [13]:
import gc

In [14]:
import csv
import os

# Create a custom log file in the output directory
log_file_path = "/kaggle/working/training_history.csv"

# Write headers if file doesn't exist
if not os.path.exists(log_file_path):
    with open(log_file_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss"])

# Inside your training loop at the end of every epoch:
def append_log(epoch, train_loss, val_loss):
    with open(log_file_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, train_loss, val_loss])

In [15]:
# === Training Loop ===
best_loss = 999.0
for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc='Training'):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = bce_dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # === Validation ===
    model.eval()
    val_loss = 0
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc='Validation'):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = bce_dice_loss(outputs, masks)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    append_log(epoch + 1, train_loss, val_loss)


    # === Scheduler + EarlyStopping ===
    scheduler.step(val_loss)
    
    if val_loss < best_loss:
        best_loss = val_loss
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        save_model(model, model_path, epoch, best_loss)
        patience_counter = 0  # Reset counter since we improved
        print("Validation loss improved!")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{EarlyStopPatience}")

    # Trigger early stopping
    if patience_counter >= EarlyStopPatience:
        print("Early stopping triggered. Stopping training!")
        break

    del outputs, loss
    gc.collect()
    torch.cuda.empty_cache()

print("Training completed!")



Epoch [1/100]


Validation: 100%|██████████| 69/69 [00:35<00:00,  1.95it/s]


Train Loss: 0.3881 | Val Loss: 0.3771
Validation loss improved!

Epoch [2/100]


Validation: 100%|██████████| 69/69 [00:35<00:00,  1.97it/s]


Train Loss: 0.3803 | Val Loss: 0.3848
No improvement. Patience: 1/15

Epoch [3/100]


...

Train Loss: 0.3600 | Val Loss: 0.3807
No improvement. Patience: 13/15

Epoch [46/100]


Validation: 100%|██████████| 69/69 [00:37<00:00,  1.85it/s]


Train Loss: 0.3571 | Val Loss: 0.3644
No improvement. Patience: 14/15

Epoch [47/100]


Validation: 100%|██████████| 69/69 [00:37<00:00,  1.85it/s]

Train Loss: 0.3533 | Val Loss: 0.3679
No improvement. Patience: 15/15
Early stopping triggered. Stopping training!
Training completed!
